In [1]:
# ==========================================
# CELL 1: ENVIRONMENT & DEVICE SETUP
# ==========================================
import os
import torch

os.environ['TORCH'] = torch.__version__
print(f"PyTorch version: {torch.__version__}")

# We removed torch-cluster because it fails to build on Mac Apple Silicon with PyTorch 2.6.
# Native torch and pyg is quite enough!
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

# Standard PyTorch ecosystem
import torch.nn as nn
import torch.nn.functional as F

# PyTorch Geometric core
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T

# PyTorch Geometric utilities and layers
from torch_geometric.utils import erdos_renyi_graph
from torch_geometric.nn import GCNConv, global_mean_pool

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Active Device for Experiment: {device}")

PyTorch version: 2.6.0
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "/opt/anaconda3/envs/cs181/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "/opt/anaconda3/envs/cs181/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "/opt/anaconda3/envs/cs181/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "/private/var/folders/r6/p7fjysz958dc3sw6yhf9m8540000gn/T/pip-build-env-50012vv3/overlay/lib/python3.13/site-pa

In [2]:
# ==========================================
# CELL 2: EDGE-DP KL-PERTURBED DICTIONARY PIPELINE (HIGHER-ORDER QUERY VERSION)
# ==========================================
from torch_geometric.utils import subgraph, k_hop_subgraph, to_undirected, coalesce, dropout_edge

# 1. Load benchmark graph
print("Loading Cora Dataset...")
dataset = Planetoid(root='/tmp/Cora', name='Cora', transform=T.NormalizeFeatures())
data = dataset[0]

# --- Setup: Public / Private / Validation split ---
num_public = int(0.2 * data.num_nodes)
num_val = int(0.2 * data.num_nodes)
perm_idx = torch.randperm(data.num_nodes)
public_nodes = perm_idx[:num_public]
val_nodes = perm_idx[num_public:num_public + num_val]
train_nodes = perm_idx[num_public + num_val:]
print(f"Dataset Split: {len(public_nodes)} Public | {len(train_nodes)} Private Train | {len(val_nodes)} Validation")

# Inductive isolation: keep split-specific edge sets
pub_edge_index, _ = subgraph(public_nodes, data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)
private_edge_index, _ = subgraph(train_nodes, data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)
val_edge_index, _ = subgraph(val_nodes, data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)

# --- Engineering check: strict L2 normalization ---
x_l2 = F.normalize(data.x, p=2, dim=1)
max_norm = float(torch.linalg.norm(x_l2, ord=2, dim=1).max().item())
print(f"Max node feature L2 norm after normalization: {max_norm:.6f}")


def one_hop_mean_aggregate(edge_index, x_features, num_nodes):
    """Compute H_u = mean_{v in N(u)} x_v on an arbitrary edge set."""
    edge_undirected = to_undirected(edge_index, num_nodes=num_nodes)
    edge_undirected = coalesce(edge_undirected, num_nodes=num_nodes)

    H_sum = torch.zeros_like(x_features)
    if edge_undirected.numel() == 0:
        return H_sum

    row, col = edge_undirected
    H_sum.scatter_add_(0, row.unsqueeze(1).expand(-1, x_features.size(1)), x_features[col])

    counts = torch.zeros((num_nodes,), dtype=x_features.dtype, device=x_features.device)
    counts.scatter_add_(0, row, torch.ones_like(row, dtype=x_features.dtype))

    H_mean = torch.zeros_like(H_sum)
    nonzero = counts > 0
    H_mean[nonzero] = H_sum[nonzero] / counts[nonzero].unsqueeze(1)
    return H_mean


def degree_weighted_sgc_embedding(x_sub, edge_index_sub):
    """Degree-weighted graph embedding for one candidate subgraph."""
    num_sub_nodes = x_sub.size(0)
    if num_sub_nodes == 0:
        return torch.zeros(data.num_features, device=x_sub.device)

    if edge_index_sub.numel() == 0:
        weights = torch.ones(num_sub_nodes, dtype=x_sub.dtype, device=x_sub.device)
    else:
        row, col = edge_index_sub
        deg = torch.bincount(row, minlength=num_sub_nodes).to(x_sub.dtype)
        deg = deg + torch.bincount(col, minlength=num_sub_nodes).to(x_sub.dtype)
        weights = deg + 1.0

    weighted_sum = (x_sub * weights.unsqueeze(1)).sum(dim=0)
    return weighted_sum / weights.sum().clamp_min(1e-12)


def build_prototype_feature(edge_index_sub, x_sub, query_mode="one_hop"):
    """
    Build graph-level prototype feature aligned with query feature design.

    one_hop:       z = normalize(pool(H1_nodes))
    two_hop_concat:z = normalize([pool(H1_nodes) || pool(H2_nodes)])
    """
    num_sub_nodes = x_sub.size(0)
    H1_nodes = one_hop_mean_aggregate(edge_index_sub, x_sub, num_nodes=num_sub_nodes)
    z1 = degree_weighted_sgc_embedding(H1_nodes, edge_index_sub)

    if query_mode == "one_hop":
        return F.normalize(z1, p=2, dim=0)

    if query_mode == "two_hop_concat":
        H2_nodes = one_hop_mean_aggregate(edge_index_sub, H1_nodes, num_nodes=num_sub_nodes)
        z2 = degree_weighted_sgc_embedding(H2_nodes, edge_index_sub)
        return F.normalize(torch.cat([z1, z2], dim=0), p=2, dim=0)

    raise ValueError(f"Unsupported query_mode: {query_mode}")


def build_private_query_features(private_edges, x_features, query_mode="one_hop"):
    """
    Build per-node query vectors from private graph structure.

    one_hop:        q_u = normalize(H1[u])
    two_hop_concat: q_u = normalize([H1[u] || H2[u]])
    """
    num_nodes = x_features.size(0)
    H1 = one_hop_mean_aggregate(private_edges, x_features, num_nodes=num_nodes)

    if query_mode == "one_hop":
        return F.normalize(H1, p=2, dim=1)

    if query_mode == "two_hop_concat":
        H2 = one_hop_mean_aggregate(private_edges, H1, num_nodes=num_nodes)
        Q = torch.cat([H1, H2], dim=1)
        return F.normalize(Q, p=2, dim=1)

    raise ValueError(f"Unsupported query_mode: {query_mode}")


def build_class_conditional_public_dictionary(
    data_obj,
    x_features,
    labels_tensor,
    public_nodes_tensor,
    pub_edges,
    num_classes,
    dict_per_class=50,
    num_hops=2,
    query_mode="one_hop",
    min_class_fraction=1.0,
):
    """
    Build class-conditional dictionary from public graph.

    min_class_fraction:
      - 1.0 keeps strict class-pure prototypes.
      - <1.0 relaxes purity by keeping enough majority-class nodes plus minority context.
    """
    print(
        f"Building class-conditional public dictionary "
        f"({dict_per_class} per class, hops={num_hops}, mode={query_mode}, min_class_fraction={min_class_fraction:.2f})..."
    )
    if public_nodes_tensor.numel() == 0:
        raise ValueError("Public split is empty, cannot build dictionary.")

    pub_undirected = to_undirected(pub_edges, num_nodes=data_obj.num_nodes)
    pub_undirected = coalesce(pub_undirected, num_nodes=data_obj.num_nodes)

    public_labels = labels_tensor[public_nodes_tensor]
    dictionary = []
    class_to_proto_indices = {c: [] for c in range(num_classes)}
    proto_purities = []

    for c in range(num_classes):
        class_pool = public_nodes_tensor[public_labels == c]
        if class_pool.numel() == 0:
            class_pool = public_nodes_tensor

        for _ in range(dict_per_class):
            anchor = class_pool[torch.randint(0, class_pool.numel(), (1,)).item()]
            subset, _, _, _ = k_hop_subgraph(
                int(anchor.item()),
                num_hops=num_hops,
                edge_index=pub_undirected,
                relabel_nodes=False,
                num_nodes=data_obj.num_nodes,
            )

            proto_labels = labels_tensor[subset]
            class_mask = proto_labels == c
            class_count = int(class_mask.sum().item())
            subset_count = int(subset.numel())

            if subset_count == 0:
                class_subset = anchor.view(1)
            else:
                frac = class_count / max(subset_count, 1)
                if frac >= min_class_fraction and class_count > 0:
                    class_subset = subset
                else:
                    # Fallback to strict class subset if relaxed purity threshold is not met.
                    class_subset = subset[class_mask]
                    if class_subset.numel() == 0:
                        class_subset = anchor.view(1)

            sub_edge_index, _ = subgraph(
                class_subset,
                pub_undirected,
                relabel_nodes=True,
                num_nodes=data_obj.num_nodes,
            )

            x_sub = x_features[class_subset]
            proto_feat = build_prototype_feature(sub_edge_index, x_sub, query_mode=query_mode)

            proto_subset_labels = labels_tensor[class_subset]
            purity = float((proto_subset_labels == c).float().mean().item())
            proto_purities.append(purity)

            proto_idx = len(dictionary)
            dictionary.append({
                'edge_index': sub_edge_index,
                'x': x_sub,
                'proto_feat': proto_feat,
                'class': c,
                'purity': purity,
            })
            class_to_proto_indices[c].append(proto_idx)

    dict_feats = torch.stack([g['proto_feat'] for g in dictionary], dim=0)
    mean_purity = float(sum(proto_purities) / max(len(proto_purities), 1))
    print(f"Dictionary size={len(dictionary)} | Mean prototype purity={mean_purity:.4f} | Feature dim={dict_feats.size(1)}")
    return dictionary, dict_feats, class_to_proto_indices


def gumbel_max_sample(logits):
    """Sample argmax(logits + Gumbel(0,1)) for exact categorical sampling."""
    u = torch.rand_like(logits).clamp_(1e-8, 1 - 1e-8)
    gumbel = -torch.log(-torch.log(u))
    return torch.argmax(logits + gumbel).item()


def synthesize_edge_dp_dataset(
    private_edges,
    x_features,
    labels,
    private_nodes_tensor,
    public_dict,
    dict_features,
    class_to_proto_indices,
    epsilon_total,
    utility_sensitivity=None,
    query_mode="one_hop",
    label_conditioning=True,
):
    """
    Edge-DP synthesizer with Exponential Mechanism over class-conditional dictionary.

    query_mode:
      - one_hop
      - two_hop_concat

    label_conditioning:
      - True: candidates restricted by true label (valid only if labels are public).
      - False: candidates are full dictionary (safer if labels are private).
    """
    if utility_sensitivity is None:
        utility_sensitivity = 1.0 if query_mode == "one_hop" else 2.0

    # Edge-level composition under 1-hop aggregation affects at most two node queries.
    epsilon_per_node = epsilon_total / 2.0
    print(
        f"Synthesizing with Edge-DP Exponential Mechanism (Neg-L2): "
        f"mode={query_mode}, label_conditioning={label_conditioning}, "
        f"epsilon_total={epsilon_total:.3f}, epsilon_per_node={epsilon_per_node:.3f}, Delta_s={utility_sensitivity:.3f}"
    )

    Q = build_private_query_features(private_edges, x_features, query_mode=query_mode)
    synthetic_dataset = []

    num_classes = int(labels.max().item()) + 1
    K_global = dict_features.size(0)
    selection_counts = torch.zeros(K_global, dtype=torch.long)
    proto_label_counts = torch.zeros((K_global, num_classes), dtype=torch.long)
    entropies = []
    normalized_entropies = []
    max_probs = []

    for u in private_nodes_tensor.tolist():
        y_u = int(labels[u].item())

        if label_conditioning:
            candidate_indices = class_to_proto_indices[y_u]
            if len(candidate_indices) == 0:
                candidate_indices = list(range(K_global))
        else:
            candidate_indices = list(range(K_global))

        idx_tensor = torch.tensor(candidate_indices, dtype=torch.long)
        candidate_feats = dict_features[idx_tensor]

        q_u = Q[u]

        # Utility score: negative L2 distance.
        scores = -torch.linalg.norm(candidate_feats - q_u.unsqueeze(0), dim=1)

        # Exponential mechanism: P(k) proportional to exp((epsilon * score_k) / (2 * Delta_s))
        logits = (epsilon_per_node / (2.0 * utility_sensitivity)) * scores
        probs = torch.softmax(logits, dim=0)

        entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum().item()
        entropies.append(entropy)
        norm_const = float(torch.log(torch.tensor(float(max(len(candidate_indices), 2))).float()).item())
        normalized_entropies.append(entropy / norm_const)
        max_probs.append(float(probs.max().item()))

        local_idx = gumbel_max_sample(logits)
        proto_idx = candidate_indices[local_idx]

        selection_counts[proto_idx] += 1
        proto_label_counts[proto_idx, y_u] += 1

        proto = public_dict[proto_idx]
        synthetic_dataset.append({
            'edge_index': proto['edge_index'].clone(),
            'x': proto['x'].clone(),
            'y': labels[u].view(1).clone(),
        })

    used_mask = selection_counts > 0
    used_count = int(used_mask.sum().item())
    conflict_mask = (proto_label_counts > 0).sum(dim=1) > 1
    used_conflicts = int((conflict_mask & used_mask).sum().item())

    purity_values = []
    for k in torch.where(used_mask)[0].tolist():
        total_k = int(proto_label_counts[k].sum().item())
        if total_k > 0:
            purity_values.append(float(proto_label_counts[k].max().item()) / float(total_k))

    topk = min(10, K_global)
    top_counts, top_idx = torch.topk(selection_counts, k=topk)

    top_label_hists = {}
    for idx in top_idx.tolist():
        count = int(selection_counts[idx].item())
        if count == 0:
            continue
        top_label_hists[int(idx)] = proto_label_counts[idx].tolist()

    diagnostics = {
        'query_mode': query_mode,
        'label_conditioning': bool(label_conditioning),
        'epsilon_per_node': float(epsilon_per_node),
        'utility_sensitivity': float(utility_sensitivity),
        'dict_size': int(K_global),
        'mean_entropy': float(sum(entropies) / max(len(entropies), 1)),
        'mean_entropy_over_logk': float(sum(normalized_entropies) / max(len(normalized_entropies), 1)),
        'mean_top_probability': float(sum(max_probs) / max(len(max_probs), 1)),
        'unique_selected_prototypes': used_count,
        'unique_selected_ratio': float(used_count) / float(K_global),
        'conflicting_used_prototypes': used_conflicts,
        'conflicting_used_ratio': float(used_conflicts) / float(max(used_count, 1)),
        'mean_proto_purity': float(sum(purity_values) / max(len(purity_values), 1)),
        'top_proto_indices': [int(i) for i in top_idx.tolist()],
        'top_proto_counts': [int(c) for c in top_counts.tolist()],
        'top_proto_label_histograms': top_label_hists,
    }

    return synthetic_dataset, diagnostics


def print_synthesis_diagnostics(diag, title):
    print(f"\n--- {title} ---")
    print(
        f"Mode={diag['query_mode']} | label_conditioning={diag['label_conditioning']} "
        f"| Delta_s={diag['utility_sensitivity']:.3f}"
    )
    print(f"Mean selection entropy: {diag['mean_entropy']:.4f} (normalized: {diag['mean_entropy_over_logk']:.4f})")
    print(f"Mean top-1 selection probability: {diag['mean_top_probability']:.4f}")
    print(f"Unique selected prototypes: {diag['unique_selected_prototypes']}/{diag['dict_size']} ({diag['unique_selected_ratio']:.2%})")
    print(f"Used prototypes with label conflict: {diag['conflicting_used_prototypes']} ({diag['conflicting_used_ratio']:.2%})")
    print(f"Mean prototype purity: {diag['mean_proto_purity']:.4f}")
    print("Top prototype usage counts:")
    for idx, count in zip(diag['top_proto_indices'], diag['top_proto_counts']):
        if count > 0:
            print(f"  Prototype {idx:02d}: {count}")


# --- Execute Edge-DP Synthesis ---
num_classes = dataset.num_classes
walk_hops = 2
query_hops = 1

# Dictionary size: total K = num_classes * dict_per_class
dict_per_class = 50

# Main run (private)
epsilon_total = 1.0

# Sanity check run (effectively no privacy noise)
run_sanity_check = True
epsilon_sanity = 10000.0

# User-requested augmentation during training
feature_jitter_std = 0.05
edge_dropout_p = 0.10

# --- New controls for higher-order query experiments ---
query_mode = "two_hop_concat"  # "one_hop" or "two_hop_concat"
label_conditioning = True       # Set False if labels are private in your threat model.
min_class_fraction = 1.0        # Keep 1.0 for strict purity; lower to relax purity.
utility_sensitivity = 2.0 if query_mode == "two_hop_concat" else 1.0

public_dict, public_proto_feats, class_to_proto_indices = build_class_conditional_public_dictionary(
    data_obj=data,
    x_features=x_l2,
    labels_tensor=data.y,
    public_nodes_tensor=public_nodes,
    pub_edges=pub_edge_index,
    num_classes=num_classes,
    dict_per_class=dict_per_class,
    num_hops=walk_hops,
    query_mode=query_mode,
    min_class_fraction=min_class_fraction,
)

synthetic_data, synth_diag = synthesize_edge_dp_dataset(
    private_edges=private_edge_index,
    x_features=x_l2,
    labels=data.y,
    private_nodes_tensor=train_nodes,
    public_dict=public_dict,
    dict_features=public_proto_feats,
    class_to_proto_indices=class_to_proto_indices,
    epsilon_total=epsilon_total,
    utility_sensitivity=utility_sensitivity,
    query_mode=query_mode,
    label_conditioning=label_conditioning,
)

print(f"Generated {len(synthetic_data)} synthetic graphs (one per private node).")
print_synthesis_diagnostics(synth_diag, "Synthesis diagnostics (epsilon=1.0)")

# Majority-class baseline on validation labels
val_labels = data.y[val_nodes]
val_counts = torch.bincount(val_labels, minlength=dataset.num_classes)
majority_label = int(torch.argmax(val_counts).item())
majority_acc = float((val_labels == majority_label).float().mean().item())
print(f"Validation majority-class baseline acc: {majority_acc:.4f} (class={majority_label})")


# --- Validation subgraphs from held-out real validation split ---
print("\n--- Extracting real validation subgraphs ---")
val_undirected = to_undirected(val_edge_index, num_nodes=data.num_nodes)
val_undirected = coalesce(val_undirected, num_nodes=data.num_nodes)

val_data_list = []
for node in val_nodes.tolist():
    subset, sub_edge_index, _, _ = k_hop_subgraph(
        node,
        num_hops=query_hops,
        edge_index=val_undirected,
        relabel_nodes=True,
        num_nodes=data.num_nodes,
    )
    val_data_list.append(Data(x=x_l2[subset], edge_index=sub_edge_index, y=data.y[node].unsqueeze(0)))

val_loader = DataLoader(val_data_list, batch_size=32, shuffle=False)


# --- Train GNN on synthetic graphs (post-processing, no extra privacy cost) ---
class StandardGCN(nn.Module):
    def __init__(self, hidden_channels, num_features, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = global_mean_pool(x, batch)
        return self.lin(x)


def train_gnn(
    synthetic_dataset,
    val_loader_obj,
    num_features,
    num_classes,
    epochs=50,
    feature_jitter_std=0.05,
    edge_dropout_p=0.10,
):
    print("\n--- Training GNN on synthetic data ---")
    print(f"Training augmentations: feature_jitter_std={feature_jitter_std}, edge_dropout_p={edge_dropout_p}")

    data_list = [Data(x=d['x'], edge_index=d['edge_index'], y=d['y']) for d in synthetic_dataset]
    loader = DataLoader(data_list, batch_size=32, shuffle=True)

    model = StandardGCN(hidden_channels=16, num_features=num_features, num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        train_correct = 0

        for batch in loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            # Zero-cost post-processing augmentation on already DP-safe synthetic data.
            x_aug = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            edge_index_aug, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if edge_index_aug.numel() == 0:
                edge_index_aug = batch.edge_index

            out = model(x_aug, edge_index_aug, batch.batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch.num_graphs
            train_correct += int((out.argmax(dim=1) == batch.y).sum())

        train_acc = train_correct / len(loader.dataset)

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for val_batch in val_loader_obj:
                val_batch = val_batch.to(device)
                val_out = model(val_batch.x, val_batch.edge_index, val_batch.batch)
                val_pred = val_out.argmax(dim=1)
                val_correct += int((val_pred == val_batch.y).sum())

        val_acc = val_correct / len(val_loader_obj.dataset)
        best_val_acc = max(best_val_acc, val_acc)

        if epoch % 10 == 0 or epoch == 1:
            avg_loss = total_loss / len(loader.dataset)
            print(f"Epoch {epoch:03d} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    print(f"Training complete. Best Val Acc: {best_val_acc:.4f}")
    return model, best_val_acc


print("\n=== Main private run (epsilon=1.0) ===")
dp_model, best_val_acc_private = train_gnn(
    synthetic_dataset=synthetic_data,
    val_loader_obj=val_loader,
    num_features=data.x.size(1),
    num_classes=dataset.num_classes,
    epochs=50,
    feature_jitter_std=feature_jitter_std,
    edge_dropout_p=edge_dropout_p,
)

if run_sanity_check:
    print("\n=== Sanity check run (epsilon=10000) ===")
    synthetic_data_sanity, synth_diag_sanity = synthesize_edge_dp_dataset(
        private_edges=private_edge_index,
        x_features=x_l2,
        labels=data.y,
        private_nodes_tensor=train_nodes,
        public_dict=public_dict,
        dict_features=public_proto_feats,
        class_to_proto_indices=class_to_proto_indices,
        epsilon_total=epsilon_sanity,
        utility_sensitivity=utility_sensitivity,
        query_mode=query_mode,
        label_conditioning=label_conditioning,
    )

    print_synthesis_diagnostics(synth_diag_sanity, "Synthesis diagnostics (epsilon=10000)")

    dp_model_sanity, best_val_acc_sanity = train_gnn(
        synthetic_dataset=synthetic_data_sanity,
        val_loader_obj=val_loader,
        num_features=data.x.size(1),
        num_classes=dataset.num_classes,
        epochs=50,
        feature_jitter_std=feature_jitter_std,
        edge_dropout_p=edge_dropout_p,
    )

    print("\n=== Summary ===")
    print(f"Best Val Acc | epsilon=1.0: {best_val_acc_private:.4f}")
    print(f"Best Val Acc | epsilon=10000: {best_val_acc_sanity:.4f}")

    acc_gap = best_val_acc_sanity - best_val_acc_private
    print(f"Sanity gap (epsilon=10000 - epsilon=1.0): {acc_gap:+.4f}")

    if abs(acc_gap) <= 0.02:
        print("Interpretation: representation/matching pipeline is likely the bottleneck (DP noise is not dominant).")
    elif acc_gap > 0.02:
        print("Interpretation: DP noise contributes materially to the performance gap.")
    else:
        print("Interpretation: high-epsilon run underperformed unexpectedly; repeat across seeds for stability checks.")

Loading Cora Dataset...
Dataset Split: 541 Public | 1626 Private Train | 541 Validation
Max node feature L2 norm after normalization: 1.000000
Building class-conditional public dictionary (50 per class, hops=2, mode=two_hop_concat, min_class_fraction=1.00)...
Dictionary size=350 | Mean prototype purity=1.0000 | Feature dim=2866
Synthesizing with Edge-DP Exponential Mechanism (Neg-L2): mode=two_hop_concat, label_conditioning=True, epsilon_total=1.000, epsilon_per_node=0.500, Delta_s=2.000
Generated 1626 synthetic graphs (one per private node).

--- Synthesis diagnostics (epsilon=1.0) ---
Mode=two_hop_concat | label_conditioning=True | Delta_s=2.000
Mean selection entropy: 3.9116 (normalized: 0.9999)
Mean top-1 selection probability: 0.0205
Unique selected prototypes: 341/350 (97.43%)
Used prototypes with label conflict: 0 (0.00%)
Mean prototype purity: 1.0000
Top prototype usage counts:
  Prototype 199: 17
  Prototype 196: 15
  Prototype 176: 15
  Prototype 171: 15
  Prototype 172: 14
 

2026-04-17 12:13:27.071 python[3614:47573] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-3614-2026-04-17_12_13_26-2733953187‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2026-04-17 12:13:27.158 python[3614:47573] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-3614-2026-04-17_12_13_27-4169240084‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2026-04-17 12:13:27.181 python[3614:47573] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-3614-2026-04-17_12_13_27-475353638‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2026-04-17 12:13:27.266 python[3614:48149] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-3614-2026-04-17_12_13_27-2540345766‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2026-

Epoch 020 | Train Loss: 0.0499 | Train Acc: 0.9840 | Val Acc: 0.5564
Epoch 030 | Train Loss: 0.0456 | Train Acc: 0.9859 | Val Acc: 0.5379
Epoch 040 | Train Loss: 0.0487 | Train Acc: 0.9852 | Val Acc: 0.5896
Epoch 050 | Train Loss: 0.0658 | Train Acc: 0.9760 | Val Acc: 0.5527
Training complete. Best Val Acc: 0.6063

=== Summary ===
Best Val Acc | epsilon=1.0: 0.7098
Best Val Acc | epsilon=10000: 0.6063
Sanity gap (epsilon=10000 - epsilon=1.0): -0.1035
Interpretation: high-epsilon run underperformed unexpectedly; repeat across seeds for stability checks.


In [3]:
# ==========================================
# CELL 3: EDGE-DP CONTINUOUS BASELINE (NOISY SGC)
# ==========================================
print("\n" + "=" * 50)
print("RUNNING EDGE-DP BASELINE: NOISY SGC")
print("=" * 50)


def run_edge_dp_sgc_baseline(
    data_obj,
    private_edges,
    val_edges,
    train_nodes_idx,
    val_nodes_idx,
    epsilon=1.0,
    delta=1e-5,
    epochs=50,
):
    """
    Edge-DP baseline:
    1) 1-hop feature aggregation on private edges
    2) Gaussian perturbation on aggregated features
    3) MLP training on noisy features
    """
    num_nodes = data_obj.num_nodes
    num_features = data_obj.x.size(1)

    # L2-normalized features so edge-level contribution is bounded.
    x_norm = F.normalize(data_obj.x, p=2, dim=1)

    private_undirected = to_undirected(private_edges, num_nodes=num_nodes)
    private_undirected = coalesce(private_undirected, num_nodes=num_nodes)

    print("Pre-computing private 1-hop SGC aggregations...")
    x_agg_train = torch.zeros_like(x_norm)
    if private_undirected.numel() > 0:
        row, col = private_undirected
        x_agg_train.scatter_add_(0, row.unsqueeze(1).expand(-1, num_features), x_norm[col])

    # Conservative global L2 sensitivity for full release of all node aggregates.
    delta_l2 = 2.0
    sigma = (delta_l2 / epsilon) * torch.sqrt(torch.tensor(2.0 * torch.log(torch.tensor(1.25 / delta))))

    print(f"Adding Gaussian noise with sigma={sigma.item():.4f} (epsilon={epsilon}, delta={delta})")
    x_noisy_train = x_agg_train + torch.randn_like(x_agg_train) * sigma

    # Validation aggregation on held-out validation structure (clean)
    val_undirected = to_undirected(val_edges, num_nodes=num_nodes)
    val_undirected = coalesce(val_undirected, num_nodes=num_nodes)

    print("Pre-computing clean validation aggregations...")
    x_agg_val = torch.zeros_like(x_norm)
    if val_undirected.numel() > 0:
        val_row, val_col = val_undirected
        x_agg_val.scatter_add_(0, val_row.unsqueeze(1).expand(-1, num_features), x_norm[val_col])

    class DPMLP(nn.Module):
        def __init__(self, in_channels, hidden, out_channels):
            super().__init__()
            self.lin1 = nn.Linear(in_channels, hidden)
            self.lin2 = nn.Linear(hidden, out_channels)

        def forward(self, x):
            x = F.relu(self.lin1(x))
            return self.lin2(x)

    model = DPMLP(num_features, 16, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    train_x = x_noisy_train[train_nodes_idx].to(device)
    train_y = data_obj.y[train_nodes_idx].to(device)

    val_x = x_agg_val[val_nodes_idx].to(device)
    val_y = data_obj.y[val_nodes_idx].to(device)

    print("Training Edge-DP baseline MLP...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(train_x)
        loss = criterion(out, train_y)
        loss.backward()
        optimizer.step()

        train_pred = out.argmax(dim=1)
        train_acc = int((train_pred == train_y).sum()) / len(train_y)

        model.eval()
        with torch.no_grad():
            val_out = model(val_x)
            val_pred = val_out.argmax(dim=1)
            val_acc = int((val_pred == val_y).sum()) / len(val_y)

        if epoch % 50 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Train Loss: {loss.item():.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    print("Edge-DP baseline complete!")
    return model


baseline_model = run_edge_dp_sgc_baseline(
    data_obj=data,
    private_edges=private_edge_index,
    val_edges=val_edge_index,
    train_nodes_idx=train_nodes,
    val_nodes_idx=val_nodes,
    epsilon=1.0,
    delta=1e-5,
    epochs=50,
)


RUNNING EDGE-DP BASELINE: NOISY SGC
Pre-computing private 1-hop SGC aggregations...


/var/folders/r6/p7fjysz958dc3sw6yhf9m8540000gn/T/ipykernel_3614/1238029323.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma = (delta_l2 / epsilon) * torch.sqrt(torch.tensor(2.0 * torch.log(torch.tensor(1.25 / delta))))


Adding Gaussian noise with sigma=9.6896 (epsilon=1.0, delta=1e-05)
Pre-computing clean validation aggregations...
Training Edge-DP baseline MLP...
Epoch 001 | Train Loss: 3.1658 | Train Acc: 0.1390 | Val Acc: 0.1553
Epoch 050 | Train Loss: 0.0042 | Train Acc: 0.9982 | Val Acc: 0.1553
Edge-DP baseline complete!


In [ ]:
# ==========================================
# CELL 4: COMPACT FULL ABLATION SUITE (OPTIMIZED)
# ==========================================
import random
import time
import shutil
import itertools

try:
    import numpy as np
except Exception:
    np = None

try:
    import pandas as pd
except Exception:
    pd = None


# --- Small utility helpers ---
def set_all_seeds(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    if np is not None:
        np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def maybe_warn_disk_space(min_free_gb=8):
    free_gb = shutil.disk_usage("/").free / (1024 ** 3)
    print(f"Free disk space: {free_gb:.2f} GB")
    if free_gb < min_free_gb:
        print(
            f"[Warning] Low disk space (<{min_free_gb} GB). "
            "This can destabilize MPS/CUDA graph caching; Colab run is recommended."
        )


# --- Memory-efficient synthetic dataset representation ---
class PrototypeAssignmentDataset(torch.utils.data.Dataset):
    """Stores only selected prototype indices + labels, avoiding repeated prototype tensor cloning."""

    def __init__(self, public_dict, proto_indices, labels):
        self.public_dict = public_dict
        self.proto_indices = proto_indices.long().cpu()
        self.labels = labels.long().cpu()

    def __len__(self):
        return self.proto_indices.numel()

    def __getitem__(self, idx):
        proto_idx = int(self.proto_indices[idx].item())
        proto = self.public_dict[proto_idx]
        return Data(
            x=proto['x'],
            edge_index=proto['edge_index'],
            y=self.labels[idx].view(1),
        )


@torch.no_grad()
def synthesize_edge_dp_assignments(
    private_edges,
    x_features,
    labels,
    private_nodes_tensor,
    public_dict,
    dict_features,
    class_to_proto_indices,
    epsilon_total,
    utility_sensitivity=None,
    query_mode="one_hop",
    label_conditioning=True,
):
    """
    Optimized synthesizer:
    - Returns prototype assignments instead of materializing full synthetic graph tensors per private node.
    - Reuses candidate index tensors for lower overhead inside the sampling loop.
    """
    if utility_sensitivity is None:
        utility_sensitivity = 1.0 if query_mode == "one_hop" else 2.0

    epsilon_per_node = epsilon_total / 2.0

    Q = build_private_query_features(private_edges, x_features, query_mode=query_mode)

    num_classes_local = int(labels.max().item()) + 1
    K_global = dict_features.size(0)
    all_idx = torch.arange(K_global, dtype=torch.long)

    class_idx_tensors = {}
    for c in range(num_classes_local):
        idx_list = class_to_proto_indices.get(c, [])
        class_idx_tensors[c] = torch.tensor(idx_list, dtype=torch.long) if len(idx_list) > 0 else all_idx

    N = int(private_nodes_tensor.numel())
    selected_proto_indices = torch.empty(N, dtype=torch.long)
    private_labels = labels[private_nodes_tensor].long().clone()

    selection_counts = torch.zeros(K_global, dtype=torch.long)
    proto_label_counts = torch.zeros((K_global, num_classes_local), dtype=torch.long)
    entropies = []
    normalized_entropies = []
    max_probs = []

    for j, u in enumerate(private_nodes_tensor.tolist()):
        y_u = int(labels[u].item())

        if label_conditioning:
            candidate_idx = class_idx_tensors[y_u]
        else:
            candidate_idx = all_idx

        candidate_feats = dict_features.index_select(0, candidate_idx)
        q_u = Q[u]

        scores = -torch.linalg.norm(candidate_feats - q_u.unsqueeze(0), dim=1)
        logits = (epsilon_per_node / (2.0 * utility_sensitivity)) * scores
        probs = torch.softmax(logits, dim=0)

        entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum().item()
        entropies.append(entropy)
        norm_const = float(torch.log(torch.tensor(float(max(candidate_idx.numel(), 2))).float()).item())
        normalized_entropies.append(entropy / norm_const)
        max_probs.append(float(probs.max().item()))

        local_idx = gumbel_max_sample(logits)
        proto_idx = int(candidate_idx[local_idx].item())

        selected_proto_indices[j] = proto_idx
        selection_counts[proto_idx] += 1
        proto_label_counts[proto_idx, y_u] += 1

    used_mask = selection_counts > 0
    used_count = int(used_mask.sum().item())
    conflict_mask = (proto_label_counts > 0).sum(dim=1) > 1
    used_conflicts = int((conflict_mask & used_mask).sum().item())

    purity_values = []
    for k in torch.where(used_mask)[0].tolist():
        total_k = int(proto_label_counts[k].sum().item())
        if total_k > 0:
            purity_values.append(float(proto_label_counts[k].max().item()) / float(total_k))

    topk = min(10, K_global)
    top_counts, top_idx = torch.topk(selection_counts, k=topk)

    diagnostics = {
        'query_mode': query_mode,
        'label_conditioning': bool(label_conditioning),
        'epsilon_per_node': float(epsilon_per_node),
        'utility_sensitivity': float(utility_sensitivity),
        'dict_size': int(K_global),
        'mean_entropy': float(sum(entropies) / max(len(entropies), 1)),
        'mean_entropy_over_logk': float(sum(normalized_entropies) / max(len(normalized_entropies), 1)),
        'mean_top_probability': float(sum(max_probs) / max(len(max_probs), 1)),
        'unique_selected_prototypes': used_count,
        'unique_selected_ratio': float(used_count) / float(K_global),
        'conflicting_used_prototypes': used_conflicts,
        'conflicting_used_ratio': float(used_conflicts) / float(max(used_count, 1)),
        'mean_proto_purity': float(sum(purity_values) / max(len(purity_values), 1)),
        'top_proto_indices': [int(i) for i in top_idx.tolist()],
        'top_proto_counts': [int(c) for c in top_counts.tolist()],
    }

    assignments = {
        'proto_indices': selected_proto_indices,
        'labels': private_labels,
    }
    return assignments, diagnostics


def train_gnn_from_assignments(
    assignments,
    public_dict,
    val_loader_obj,
    num_features,
    num_classes,
    epochs=20,
    batch_size=64,
    feature_jitter_std=0.05,
    edge_dropout_p=0.10,
    lr=0.01,
):
    """
    Optimized training path for ablations:
    - Avoids materializing a full duplicated synthetic graph list.
    - Uses AMP automatically on CUDA only.
    """
    dataset_obj = PrototypeAssignmentDataset(
        public_dict=public_dict,
        proto_indices=assignments['proto_indices'],
        labels=assignments['labels'],
    )
    loader = DataLoader(dataset_obj, batch_size=batch_size, shuffle=True)

    model = StandardGCN(hidden_channels=16, num_features=num_features, num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    use_cuda_amp = (device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_cuda_amp)

    best_val_acc = 0.0
    for _ in range(epochs):
        model.train()
        for batch in loader:
            batch = batch.to(device)
            optimizer.zero_grad(set_to_none=True)

            x_aug = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            edge_index_aug, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if edge_index_aug.numel() == 0:
                edge_index_aug = batch.edge_index

            if use_cuda_amp:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    out = model(x_aug, edge_index_aug, batch.batch)
                    loss = criterion(out, batch.y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x_aug, edge_index_aug, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for val_batch in val_loader_obj:
                val_batch = val_batch.to(device)
                val_out = model(val_batch.x, val_batch.edge_index, val_batch.batch)
                val_pred = val_out.argmax(dim=1)
                val_correct += int((val_pred == val_batch.y).sum())

        val_acc = val_correct / len(val_loader_obj.dataset)
        best_val_acc = max(best_val_acc, val_acc)

        if device.type == "mps":
            torch.mps.empty_cache()

    return float(best_val_acc)


# --- Build validation loader once for all ablation runs ---
def build_validation_loader_once(query_hops_local=1, batch_size=128):
    val_undirected_local = to_undirected(val_edge_index, num_nodes=data.num_nodes)
    val_undirected_local = coalesce(val_undirected_local, num_nodes=data.num_nodes)

    val_data_list_local = []
    for node in val_nodes.tolist():
        subset, sub_edge_index, _, _ = k_hop_subgraph(
            node,
            num_hops=query_hops_local,
            edge_index=val_undirected_local,
            relabel_nodes=True,
            num_nodes=data.num_nodes,
        )
        val_data_list_local.append(
            Data(x=x_l2[subset], edge_index=sub_edge_index, y=data.y[node].unsqueeze(0))
        )

    return DataLoader(val_data_list_local, batch_size=batch_size, shuffle=False)


# --- Unified experiment runner ---
def run_one_ablation_config(cfg, seed, val_loader_obj):
    set_all_seeds(seed)

    utility_sensitivity = cfg.get('utility_sensitivity', None)
    if utility_sensitivity is None:
        utility_sensitivity = 1.0 if cfg['query_mode'] == 'one_hop' else 2.0

    public_dict_local, public_proto_feats_local, class_to_proto_indices_local = build_class_conditional_public_dictionary(
        data_obj=data,
        x_features=x_l2,
        labels_tensor=data.y,
        public_nodes_tensor=public_nodes,
        pub_edges=pub_edge_index,
        num_classes=dataset.num_classes,
        dict_per_class=cfg['dict_per_class'],
        num_hops=cfg['walk_hops'],
        query_mode=cfg['query_mode'],
        min_class_fraction=cfg['min_class_fraction'],
    )

    assignments_local, diag_local = synthesize_edge_dp_assignments(
        private_edges=private_edge_index,
        x_features=x_l2,
        labels=data.y,
        private_nodes_tensor=train_nodes,
        public_dict=public_dict_local,
        dict_features=public_proto_feats_local,
        class_to_proto_indices=class_to_proto_indices_local,
        epsilon_total=cfg['epsilon_total'],
        utility_sensitivity=utility_sensitivity,
        query_mode=cfg['query_mode'],
        label_conditioning=cfg['label_conditioning'],
    )

    best_val_acc_local = train_gnn_from_assignments(
        assignments=assignments_local,
        public_dict=public_dict_local,
        val_loader_obj=val_loader_obj,
        num_features=data.x.size(1),
        num_classes=dataset.num_classes,
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        feature_jitter_std=cfg['feature_jitter_std'],
        edge_dropout_p=cfg['edge_dropout_p'],
        lr=cfg['lr'],
    )

    row = {
        'study': cfg['study'],
        'param_name': cfg['param_name'],
        'param_value': cfg['param_value'],
        'seed': int(seed),
        'best_val_acc': float(best_val_acc_local),
        'epsilon_total': float(cfg['epsilon_total']),
        'query_mode': cfg['query_mode'],
        'label_conditioning': bool(cfg['label_conditioning']),
        'dict_per_class': int(cfg['dict_per_class']),
        'min_class_fraction': float(cfg['min_class_fraction']),
        'utility_sensitivity': float(utility_sensitivity),
        'mean_entropy': float(diag_local['mean_entropy']),
        'mean_entropy_over_logk': float(diag_local['mean_entropy_over_logk']),
        'mean_top_probability': float(diag_local['mean_top_probability']),
        'unique_selected_ratio': float(diag_local['unique_selected_ratio']),
        'dict_size': int(diag_local['dict_size']),
    }
    return row


# --- Full ablation plan ---
maybe_warn_disk_space(min_free_gb=8)

ABLATION_SETTINGS = {
    'seeds': [0, 1, 2],
    'epochs': 20,
    'batch_size': 64,
    'lr': 0.01,
    'feature_jitter_std': 0.05,
    'edge_dropout_p': 0.10,
    'base': {
        'epsilon_total': 1.0,
        'query_mode': 'two_hop_concat',
        'label_conditioning': True,
        'dict_per_class': 50,
        'min_class_fraction': 1.0,
        'walk_hops': 2,
    },
    # "ALL" ablations discussed: epsilon, dictionary size, purity, label conditioning, query design.
    'epsilon_sweep': [0.25, 0.5, 1.0, 2.0, 4.0, 10000.0],
    'dict_per_class_sweep': [10, 25, 50, 100],
    'min_class_fraction_sweep': [1.0, 0.8, 0.6],
    'label_conditioning_sweep': [True, False],
    'query_mode_sweep': ['one_hop', 'two_hop_concat'],
}


def build_ablation_configs(settings):
    base = dict(settings['base'])
    cfgs = []

    for eps in settings['epsilon_sweep']:
        cfg = dict(base)
        cfg.update({'study': 'epsilon_sweep', 'param_name': 'epsilon_total', 'param_value': eps, 'epsilon_total': eps})
        cfgs.append(cfg)

    for dpc in settings['dict_per_class_sweep']:
        cfg = dict(base)
        cfg.update({'study': 'dict_size_sweep', 'param_name': 'dict_per_class', 'param_value': dpc, 'dict_per_class': dpc})
        cfgs.append(cfg)

    for purity in settings['min_class_fraction_sweep']:
        cfg = dict(base)
        cfg.update({'study': 'purity_sweep', 'param_name': 'min_class_fraction', 'param_value': purity, 'min_class_fraction': purity})
        cfgs.append(cfg)

    for lc in settings['label_conditioning_sweep']:
        cfg = dict(base)
        cfg.update({'study': 'label_conditioning_sweep', 'param_name': 'label_conditioning', 'param_value': lc, 'label_conditioning': lc})
        cfgs.append(cfg)

    for qm in settings['query_mode_sweep']:
        cfg = dict(base)
        cfg.update({'study': 'query_mode_sweep', 'param_name': 'query_mode', 'param_value': qm, 'query_mode': qm})
        cfgs.append(cfg)

    return cfgs


# Run all ablations.
RUN_FULL_ABLATIONS = True

if RUN_FULL_ABLATIONS:
    start_t = time.time()
    val_loader_ablation = build_validation_loader_once(query_hops_local=1, batch_size=128)

    ablation_cfgs = build_ablation_configs(ABLATION_SETTINGS)
    seeds = ABLATION_SETTINGS['seeds']

    # Inject shared training knobs into each config.
    for cfg in ablation_cfgs:
        cfg['epochs'] = ABLATION_SETTINGS['epochs']
        cfg['batch_size'] = ABLATION_SETTINGS['batch_size']
        cfg['lr'] = ABLATION_SETTINGS['lr']
        cfg['feature_jitter_std'] = ABLATION_SETTINGS['feature_jitter_std']
        cfg['edge_dropout_p'] = ABLATION_SETTINGS['edge_dropout_p']

    total_runs = len(ablation_cfgs) * len(seeds)
    print(f"Starting full ablation suite: {len(ablation_cfgs)} configs x {len(seeds)} seeds = {total_runs} runs")

    rows = []
    run_idx = 0
    for cfg in ablation_cfgs:
        for seed in seeds:
            run_idx += 1
            print(
                f"[{run_idx:03d}/{total_runs}] "
                f"study={cfg['study']} | {cfg['param_name']}={cfg['param_value']} | seed={seed}"
            )
            row = run_one_ablation_config(cfg, seed, val_loader_ablation)
            rows.append(row)

    elapsed = time.time() - start_t
    print(f"Ablation suite complete in {elapsed / 60.0:.2f} minutes")

    if pd is not None:
        results_df = pd.DataFrame(rows)
        summary_df = (
            results_df
            .groupby(['study', 'param_name', 'param_value'], dropna=False)
            .agg(
                runs=('best_val_acc', 'count'),
                best_val_acc_mean=('best_val_acc', 'mean'),
                best_val_acc_std=('best_val_acc', 'std'),
                entropy_mean=('mean_entropy', 'mean'),
                top_prob_mean=('mean_top_probability', 'mean'),
                unique_ratio_mean=('unique_selected_ratio', 'mean'),
            )
            .reset_index()
            .sort_values(['study', 'param_value'])
        )

        results_df.to_csv('ablation_results_full.csv', index=False)
        summary_df.to_csv('ablation_summary_full.csv', index=False)

        print("Saved: ablation_results_full.csv")
        print("Saved: ablation_summary_full.csv")
        display(summary_df)
    else:
        print("pandas not available; printing first 10 raw rows instead.")
        print(rows[:10])
else:
    print("RUN_FULL_ABLATIONS=False; cell loaded utilities only.")